# Polyp Detection — Augmentation + Negative Samples

Second hypothesis test: stronger augmentation and negative samples,
following the YOLO-LAN (2025) paper which reported +15.2% mAP50:95
on the same Kvasir-SEG dataset using this approach.

After imgsz=960 failed (notebook 04c), this notebook tests whether
the small-object recall gap can be closed through training pipeline
changes rather than architectural ones.

---

## What this notebook does

1. Samples 140 polyp-free frames from Kvasir (normal categories)
   and adds them to the training split with empty label files
2. Retrains YOLOv8m-seg with stronger augmentation:
   flipud=0.5, degrees=45, hsv_s=0.9, hsv_v=0.5
3. Evaluates using the same framework as notebooks 04 and 04b
4. Compares all approaches in a summary table

## Why negative samples

Kvasir-SEG contains only polyp-positive images. Without negative
examples, the model never learns to output nothing — it develops
a bias toward always detecting something. Adding 140 polyp-free
frames (20% of training size) corrects this.

## Result

CVC-ClinicDB recall improved from 75.2% to 81.4% (+6.2 percentage points).
This became the final model configuration.

## Output

- models/aug_neg/weights/best.pt
- results/metrics/aug_neg_metrics.json
- results/figures/aug_neg_size_comparison.png

In [ ]:
# Import libraries

import json
import os
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "dataset_yaml":      os.path.join(BASE_DIR, "configs", "dataset.yaml"),
    "cvc_eval_yaml":     os.path.join(BASE_DIR, "configs", "dataset_cvc_eval.yaml"),
    "train_images":      os.path.join(BASE_DIR, "data", "yolo_format", "train", "images"),
    "train_labels":      os.path.join(BASE_DIR, "data", "yolo_format", "train", "labels"),
    "neg_raw":           os.path.join(BASE_DIR, "data", "negatives_raw"),
    "cvc_images":        os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "images"),
    "cvc_labels":        os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "labels"),
    "metrics":           os.path.join(BASE_DIR, "results", "metrics"),
    "figures":           os.path.join(BASE_DIR, "results", "figures"),
    "aug_neg_yaml":      os.path.join(BASE_DIR, "configs", "dataset_aug_neg.yaml"),
}

os.makedirs(PATHS["neg_raw"],  exist_ok=True)
os.makedirs(PATHS["metrics"],  exist_ok=True)
os.makedirs(PATHS["figures"],  exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "will be created"
    print(f"  [{status}] {name:16s} -> {path}")

In [ ]:
# Download Negative Samples
#
# We need polyp-free colonoscopy frames. The Kvasir dataset
# (NOT Kvasir-SEG) contains multiple GI tract categories,
# including normal-z-line, normal-cecum, normal-pylorus -
# these are polyp-free by definition and from the same domain.
#
# Download from: https://datasets.simula.no/kvasir/
# We only need the "normal-*" categories (3 folders, ~750 images total)
#
# Manual steps:
#   1. Download kvasir-dataset-v2.zip from the link above
#   2. Extract only these folders into data/negatives_raw/:
#      - normal-z-line/
#      - normal-cecum/
#      - normal-pylorus/
#
# After extraction, run this cell to verify

neg_categories = ["normal-z-line", "normal-cecum", "normal-pylorus"]
total_neg = 0

for cat in neg_categories:
    cat_path = os.path.join(PATHS["neg_raw"], cat)
    if os.path.exists(cat_path):
        count = len([f for f in os.listdir(cat_path)
                     if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        total_neg += count
        print(f"  {cat}: {count} images")
    else:
        print(f"  {cat}: NOT FOUND - download and extract first")

print(f"\nTotal negative samples available: {total_neg}")
print("We will use 20% of training set size (= ~140 images)")

In [ ]:
# Add Negatives to Training Split
#
# Add negative samples to the training folder.
# Empty label files tell YOLO "this image has no polyps" -
# the model learns to output nothing for these frames.
#
# We cap at 20% of training size, following the paper's ratio.
# Using more than this can hurt recall on the positive cases.

import random
random.seed(42)

n_train = len([f for f in os.listdir(PATHS["train_images"])
               if f.lower().endswith((".jpg", ".jpeg", ".png"))])
n_negatives_to_add = int(n_train * 0.20)  # ~140 images

print(f"Training set size: {n_train}")
print(f"Negatives to add:  {n_negatives_to_add} (20%)")

# Collect all available negatives
all_negatives = []
for cat in neg_categories:
    cat_path = os.path.join(PATHS["neg_raw"], cat)
    if os.path.exists(cat_path):
        for fname in os.listdir(cat_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                all_negatives.append(os.path.join(cat_path, fname))

selected = random.sample(all_negatives, min(n_negatives_to_add, len(all_negatives)))

added = 0
for src_path in selected:
    fname = "neg_" + os.path.basename(src_path)
    dst_img = os.path.join(PATHS["train_images"], fname)
    dst_lbl = os.path.join(PATHS["train_labels"],
                           os.path.splitext(fname)[0] + ".txt")

    if not os.path.exists(dst_img):
        shutil.copy2(src_path, dst_img)
        open(dst_lbl, "w").close()  # empty label = no polyp
        added += 1

print(f"Added {added} negative images to training split.")
print(f"New training size: {n_train + added}")

In [ ]:
# Write New dataset.yaml
# Write a new dataset config that uses the augmented training split.
# This is separate from the original dataset.yaml so we can always
# go back to the clean split if needed.

import yaml

with open(PATHS["dataset_yaml"]) as f:
    base_config = yaml.safe_load(f)

aug_neg_config = {
    "path":     base_config["path"],
    "train":    "train/images",       # same folder, now with negatives added
    "val":      base_config["val"],
    "test":     base_config["test"],
    "cvc_test": base_config["cvc_test"],
    "nc":       1,
    "names":    ["polyp"],
}

with open(PATHS["aug_neg_yaml"], "w") as f:
    yaml.dump(aug_neg_config, f, default_flow_style=False, sort_keys=False)

print(f"Saved -> {PATHS['aug_neg_yaml']}")

In [ ]:
# Training Hyperparameters
# Training configuration with stronger augmentation.
#
# Key differences from baseline:
#   flipud=0.5  : vertical flip (was 0.0) - endoscope rotates freely
#   degrees=45  : rotation up to 45 degrees (was 0.0), per YOLO-LAN paper
#   hsv_s=0.9   : more saturation variation (was 0.7)
#   hsv_v=0.5   : more brightness variation (was 0.4) - replaces blur
#
# Note: "blur" is not a valid Ultralytics train() parameter.
# Brightness/saturation variation (hsv_v, hsv_s) achieves similar
# texture diversity without invalid arguments.
#
# imgsz and batch stay at 640 and 8 - same as baseline, so any
# improvement is purely from augmentation + negatives, not hardware.

TRAIN_CONFIG = {
    "model":     "yolov8m-seg.pt",
    "data":      PATHS["aug_neg_yaml"],
    "epochs":    100,
    "imgsz":     640,
    "batch":     8,
    "patience":  15,
    "optimizer": "AdamW",
    "lr0":       0.001,
    "seed":      42,
    "amp":       True,
    "flipud":    0.5,
    "degrees":   45,
    "hsv_s":     0.9,
    "hsv_v":     0.5,
    "project":   os.path.join(BASE_DIR, "models"),
    "name":      "aug_neg",
    "exist_ok":  True,
    "verbose":   True,
}

print("Training configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key:10s}: {value}")

In [ ]:
# Run Training

model = YOLO(TRAIN_CONFIG["model"])

start_time = time.time()

train_results = model.train(
    data=TRAIN_CONFIG["data"],
    epochs=TRAIN_CONFIG["epochs"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    patience=TRAIN_CONFIG["patience"],
    optimizer=TRAIN_CONFIG["optimizer"],
    lr0=TRAIN_CONFIG["lr0"],
    seed=TRAIN_CONFIG["seed"],
    amp=TRAIN_CONFIG["amp"],
    flipud=TRAIN_CONFIG["flipud"],
    degrees=TRAIN_CONFIG["degrees"],
    hsv_s=TRAIN_CONFIG["hsv_s"],
    hsv_v=TRAIN_CONFIG["hsv_v"],
    project=TRAIN_CONFIG["project"],
    name=TRAIN_CONFIG["name"],
    exist_ok=TRAIN_CONFIG["exist_ok"],
    verbose=TRAIN_CONFIG["verbose"],
)

elapsed_min = (time.time() - start_time) / 60
print(f"\nTraining finished in {elapsed_min:.1f} minutes")

In [ ]:
# Load Model and Evaluate
# Evaluate using the same conf=0.30 threshold from notebook 04
# for direct comparison with baseline

weights_path = os.path.join(
    TRAIN_CONFIG["project"], TRAIN_CONFIG["name"], "weights", "best.pt"
)
model_aug = YOLO(weights_path)
print(f"Loaded: {weights_path}")

kvasir_results = model_aug.val(
    data=PATHS["dataset_yaml"],
    split="test",
    imgsz=640,
    conf=0.30,
)

cvc_results = model_aug.val(
    data=PATHS["cvc_eval_yaml"],
    split="val",
    imgsz=640,
    conf=0.30,
)

print("\nKVASIR-SEG TEST (aug+neg, conf=0.30)")
print("-" * 40)
print(f"Recall:    {kvasir_results.box.mr:.4f}")
print(f"Precision: {kvasir_results.box.mp:.4f}")
print(f"mAP50:     {kvasir_results.box.map50:.4f}")

print("\nCVC-CLINICDB CROSS-DATASET (aug+neg, conf=0.30)")
print("-" * 40)
print(f"Recall:    {cvc_results.box.mr:.4f}")
print(f"Precision: {cvc_results.box.mp:.4f}")
print(f"mAP50:     {cvc_results.box.map50:.4f}")

In [ ]:
# Error Analysis (Same as 04b)
# Re-run the missed-polyp analysis to see if the size gap shrank

def read_yolo_label_as_boxes(label_path, img_width, img_height):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            coords = list(map(float, parts[1:]))
            points = np.array(coords).reshape(-1, 2)
            points[:, 0] *= img_width
            points[:, 1] *= img_height
            x_min, y_min = points.min(axis=0)
            x_max, y_max = points.max(axis=0)
            boxes.append([x_min, y_min, x_max, y_max])
    return boxes


def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    if inter_area == 0:
        return 0.0
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter_area / (area1 + area2 - inter_area)


IOU_MATCH_THRESHOLD = 0.3
image_files = sorted(os.listdir(PATHS["cvc_images"]))

missed_aug   = []
detected_aug = []

for fname in image_files:
    stem = os.path.splitext(fname)[0]
    img_path   = os.path.join(PATHS["cvc_images"], fname)
    label_path = os.path.join(PATHS["cvc_labels"], stem + ".txt")

    img = Image.open(img_path)
    w, h = img.size
    gt_boxes = read_yolo_label_as_boxes(label_path, w, h)
    if not gt_boxes:
        continue

    pred = model_aug.predict(img_path, conf=0.10, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) > 0 else []

    for gt_box in gt_boxes:
        gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
        gt_area_ratio = gt_area / (w * h)
        best_iou = max((box_iou(gt_box, pb) for pb in pred_boxes), default=0.0)
        record = {"image": fname, "area_ratio": gt_area_ratio}
        if best_iou < IOU_MATCH_THRESHOLD:
            missed_aug.append(record)
        else:
            detected_aug.append(record)

miss_rate_aug = len(missed_aug) / (len(missed_aug) + len(detected_aug))

print(f"Total polyps:      {len(missed_aug) + len(detected_aug)}")
print(f"Missed (aug+neg):  {len(missed_aug)}")
print(f"Miss rate:         {miss_rate_aug*100:.1f}%")
print(f"Miss rate baseline (640): 11.3%")
print(f"Miss rate imgsz960:       14.1%")

In [ ]:
# Final Comparison Table
# Load baseline metrics for comparison

baseline_path = os.path.join(PATHS["metrics"], "baseline_metrics.json")
with open(baseline_path) as f:
    baseline = json.load(f)

print("=" * 60)
print("FINAL COMPARISON: ALL APPROACHES")
print("=" * 60)
print(f"{'Approach':<25} {'CVC Recall':>10} {'CVC Prec':>10} {'Miss Rate':>10}")
print("-" * 60)
print(f"{'Baseline (640)':25} "
      f"{baseline['cvc_cross_dataset']['recall']:>10.3f} "
      f"{baseline['cvc_cross_dataset']['precision']:>10.3f} "
      f"{'11.3%':>10}")
print(f"{'imgsz=960':25} "
      f"{'0.751':>10} "
      f"{'0.872':>10} "
      f"{'14.1%':>10}")
print(f"{'Aug + Negatives':25} "
      f"{cvc_results.box.mr:>10.3f} "
      f"{cvc_results.box.mp:>10.3f} "
      f"{miss_rate_aug*100:>9.1f}%")
print("=" * 60)

# Save metrics
aug_neg_metrics = {
    "model": "YOLOv8m-seg",
    "config": "aug+neg (flipud=0.5, degrees=45, blur=0.3, negatives=20%)",
    "kvasir_test": {
        "recall":    float(kvasir_results.box.mr),
        "precision": float(kvasir_results.box.mp),
        "map50":     float(kvasir_results.box.map50),
    },
    "cvc_cross_dataset": {
        "recall":    float(cvc_results.box.mr),
        "precision": float(cvc_results.box.mp),
        "map50":     float(cvc_results.box.map50),
    },
    "miss_rate": miss_rate_aug,
}

metrics_path = os.path.join(PATHS["metrics"], "aug_neg_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(aug_neg_metrics, f, indent=2)
print(f"\nSaved -> {metrics_path}")

In [ ]:
# Summary

print("NOTEBOOK 04d COMPLETE")
print("=" * 50)
print(f"CVC-ClinicDB cross-dataset recall:")
print(f"  Baseline:       {baseline['cvc_cross_dataset']['recall']:.3f}")
print(f"  imgsz=960:      0.751 (worse)")
print(f"  Aug + Negatives: {cvc_results.box.mr:.3f}")
print()

if cvc_results.box.mr > baseline["cvc_cross_dataset"]["recall"]:
    improvement = (cvc_results.box.mr - baseline["cvc_cross_dataset"]["recall"]) * 100
    print(f"Improvement over baseline: +{improvement:.1f} percentage points")
else:
    print("No improvement over baseline.")
    print("All three approaches tested - small-object detection")
    print("on cross-dataset evaluation remains the core challenge.")

print()
print("Next -> 05_evaluation.ipynb")
print("  - Final comparison of all approaches")
print("  - Select best configuration")
print("  - Generate README-ready tables and figures")